# Exploratory analysis: beta-convergence, 2004-2024

The notebook explores the dataset, it does not define it: the download lives in
`src/fetch_data.py` and the transformations in `src/etl.py`. Inference is in
`R/regressions.R`.

In [ ]:
import sys

sys.path.append("../src")

import numpy as np
import pandas as pd
import statsmodels.api as sm

from etl import build_dataset, sigma_convergence

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

data = build_dataset()
print(f"{len(data)} economies | dropped: {data.attrs['dropped_incomplete']}")
data.head(3)

## Sample composition

Not hand-picked: every World Bank member economy that is not an aggregate, has
an income classification, has at least one million inhabitants, and reports GDP
per capita in every year the periods need.

In [ ]:
print(data.groupby(["group", "income_group"]).size())
print()
print(data.groupby("group")["population"].describe()[["count", "min", "50%", "max"]])

In [ ]:
analysis_cols = [c for c in data.columns if c.startswith(("growth_", "log_y0_"))]
print(data[analysis_cols].isna().mean().to_string())
data[analysis_cols].describe().transpose().round(3)

## The regressor that matters

Growth on past growth and growth on initial income are nearly unrelated in this
sample, so the choice of regressor changes the answer rather than refining it.

In [ ]:
print("corr(growth 2008-2013, growth 2004-2008) = "
      f"{data['growth_recuperation'].corr(data['growth_pre_crisis']):.3f}")
print("corr(growth 2008-2013, log y0 2008)      = "
      f"{data['growth_recuperation'].corr(data['log_y0_recuperation']):.3f}")

In [ ]:
model = sm.OLS(data["growth_full"],
               sm.add_constant(data["log_y0_full"])).fit(cov_type="HC1")
print(model.summary().tables[1])

beta = model.params["log_y0_full"] / 100
lam = -np.log(1 + beta * 20) / 20
print()
print(f"lambda = {lam:.3%}/yr, half-life = {np.log(2) / lam:.0f} years")

## Sigma-convergence

Whether the income distribution actually narrows, which beta-convergence alone
does not guarantee.

In [ ]:
dispersion = sigma_convergence(data)
dispersion.pivot(index="year", columns="group", values="sd_log_gdp_pc").round(3)